# Accumulation de gradients : un grand batch avec peu de mémoire

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/Machine%20Learning/Accumulation-Gradients.ipynb)

📖 **Le shot complet** : [Un batch size de 128... sur un petit GPU ?](https://www.monshotdata.com/p/accumulation-gradients)

---

Un grand batch stabilise l'entraînement, mais il demande de la mémoire GPU que tu n'as pas toujours. L'**accumulation de gradients** règle ça : tu traites plusieurs petits batchs, tu additionnes leurs gradients, et tu ne mets à jour le modèle qu'ensuite. Résultat : l'effet d'un grand batch avec la mémoire d'un petit.

**À la fin de ce notebook, tu sauras :**

- comment fonctionne l'accumulation de gradients dans une boucle d'entraînement PyTorch ;
- pourquoi il faut **diviser la loss** par le nombre d'étapes d'accumulation, et ce qui se passe si on l'oublie ;
- vérifier numériquement que 4 batchs de 32 accumulés donnent exactement le même gradient qu'un batch de 128.

### Avant de lancer

- **Installation** : PyTorch et torchvision sont déjà installés sur Colab. Sinon : `pip install torch torchvision`.
- **Les données** : MNIST (chiffres manuscrits). Il est téléchargé automatiquement dans un dossier `data` au premier lancement (environ 10 Mo).
- **Durée** : environ 3 minutes sur CPU, bien moins sur GPU. Le notebook utilise le GPU s'il est disponible, sinon le CPU.
- **Les résultats** : ils varient légèrement d'une machine à l'autre, même avec des graines fixées. Compare les méthodes entre elles plutôt que les chiffres exacts.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13 et PyTorch 2.11 (CPU)). Les sorties affichées viennent de cette exécution.*

## 1. Mise en place

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import pandas as pd

from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

Calcul sur : cpu


## 2. Les données

On charge MNIST. Le **micro-batch** est de 32 images : c'est la taille que ta mémoire peut porter. Le but est de se comporter comme un batch de 128.

In [2]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

trainset = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
testset = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

testloader = DataLoader(testset, batch_size=256, shuffle=False)

def charge_train(batch_size):
    """Un chargeur d'entraînement dont l'ordre aléatoire est le même à chaque appel."""
    g = torch.Generator().manual_seed(0)
    return DataLoader(trainset, batch_size=batch_size, shuffle=True, generator=g)

print(len(trainset), "images d'entraînement,", len(testset), "images de test")

60000 images d'entraînement, 10000 images de test


## 3. Le réseau et l'évaluation

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return self.fc4(x)

def evaluate(model):
    """Précision du modèle sur le jeu de test."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            predicted = model(inputs).argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

## 4. Le principe, vérifié sur un exemple

Quand tu appelles `loss.backward()`, PyTorch **ajoute** le gradient calculé à celui déjà stocké dans chaque paramètre (`param.grad`). C'est ce qui rend l'accumulation possible : au lieu de remettre les gradients à zéro après chaque batch, on appelle `backward()` sur 4 micro-batchs de suite, puis on fait un seul `optimizer.step()`.

Mais il y a un détail. La loss de PyTorch est une **moyenne** sur le batch. Un batch de 128 donne la moyenne sur 128 images ; 4 micro-batchs de 32 donnent 4 moyennes, que `backward()` **additionne**. Pour retrouver la moyenne sur 128, il faut donc diviser chaque loss par 4.

Vérifions-le : on calcule le gradient d'un batch de 128, puis celui de 4 micro-batchs de 32 accumulés, avec et sans la division par 4.

In [4]:
torch.manual_seed(0)
modele = Net().to(device)
critere = nn.CrossEntropyLoss()

# 128 images : un grand batch, puis le même découpé en 4 micro-batchs de 32
X128 = torch.stack([trainset[i][0] for i in range(128)]).to(device)
y128 = torch.tensor([trainset[i][1] for i in range(128)]).to(device)

def gradients():
    return torch.cat([p.grad.flatten().clone() for p in modele.parameters()])

# 1. Gradient d'un seul batch de 128
modele.zero_grad()
critere(modele(X128), y128).backward()
g_grand_batch = gradients()

# 2. Gradient accumulé sur 4 micro-batchs de 32, loss divisée par 4
modele.zero_grad()
for Xb, yb in zip(X128.chunk(4), y128.chunk(4)):
    (critere(modele(Xb), yb) / 4).backward()
g_accumule = gradients()

# 3. Même chose, mais en oubliant de diviser la loss
modele.zero_grad()
for Xb, yb in zip(X128.chunk(4), y128.chunk(4)):
    critere(modele(Xb), yb).backward()
g_sans_division = gradients()

print("Écart max entre grand batch et accumulation (avec division) :", f"{(g_grand_batch - g_accumule).abs().max().item():.2e}")
print("Rapport des normes, accumulation sans division / grand batch :", f"{(g_sans_division.norm() / g_grand_batch.norm()).item():.2f}")

Écart max entre grand batch et accumulation (avec division) : 7.45e-09
Rapport des normes, accumulation sans division / grand batch : 4.00


Avec la division, le gradient accumulé est **identique** à celui du grand batch (l'écart est de l'ordre des erreurs d'arrondi). Sans la division, il est **4 fois trop grand** : chaque mise à jour est 4 fois plus violente que prévu, comme si on avait multiplié le taux d'apprentissage par 4.

## 5. Entraînement classique, micro-batch de 32

La référence : on met à jour le modèle après chaque batch de 32.

In [5]:
def entraine_classique(batch_size, epochs=2):
    torch.manual_seed(0)
    net = Net().to(device)
    critere = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)
    loader = charge_train(batch_size)

    for epoch in range(epochs):
        net.train()
        loss_totale = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = critere(net(inputs), labels)
            loss.backward()
            optimizer.step()
            loss_totale += loss.item()
        print(f"Epoch {epoch + 1}, Loss : {loss_totale / len(loader):.3f}, Précision : {evaluate(net) * 100:.2f}%")
    return evaluate(net)

acc_classique_32 = entraine_classique(batch_size=32)

Epoch 1, Loss : 0.310, Précision : 95.55%


Epoch 2, Loss : 0.147, Précision : 95.91%


## 6. Avec accumulation de gradients

Micro-batch de 32 (donc même consommation de mémoire), mais une mise à jour toutes les `accumulation_steps = 4` étapes : le **batch effectif** est de 32 × 4 = 128.

Trois points à retenir dans le code :

- la loss est **divisée** par `accumulation_steps` avant `backward()` ;
- `optimizer.step()` et `optimizer.zero_grad()` ne s'exécutent qu'une étape sur 4, et à la toute dernière étape de l'epoch pour ne pas perdre les derniers gradients ;
- la loss affichée est remultipliée par `accumulation_steps`, pour rester comparable aux autres entraînements.

In [6]:
def entraine_accumulation(batch_size=32, accumulation_steps=4, epochs=2, diviser_loss=True):
    torch.manual_seed(0)
    net = Net().to(device)
    critere = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=0.001)
    loader = charge_train(batch_size)

    for epoch in range(epochs):
        net.train()
        loss_totale = 0.0
        optimizer.zero_grad()

        for idx, (inputs, labels) in enumerate(loader):
            inputs, labels = inputs.to(device), labels.to(device)
            loss = critere(net(inputs), labels)
            loss_totale += loss.item()

            if diviser_loss:
                loss = loss / accumulation_steps   # la somme des gradients redevient une moyenne
            loss.backward()                         # le gradient s'ajoute à celui déjà accumulé

            derniere_etape = (idx + 1) == len(loader)
            if (idx + 1) % accumulation_steps == 0 or derniere_etape:
                optimizer.step()
                optimizer.zero_grad()

        print(f"Epoch {epoch + 1}, Loss : {loss_totale / len(loader):.3f}, Précision : {evaluate(net) * 100:.2f}%")
    return evaluate(net)

acc_accumulation = entraine_accumulation()

Epoch 1, Loss : 0.371, Précision : 94.62%


Epoch 2, Loss : 0.153, Précision : 95.79%


## 7. La référence : un vrai batch de 128

C'est ce que l'accumulation imite. Il demande 4 fois plus de mémoire par étape, ce que l'accumulation évite.

In [7]:
acc_classique_128 = entraine_classique(batch_size=128)

Epoch 1, Loss : 0.371, Précision : 94.79%


Epoch 2, Loss : 0.153, Précision : 95.86%


## 8. Et si on oublie de diviser la loss ?

Même entraînement qu'à la section 6, mais sans la division. C'est l'erreur la plus fréquente.

In [8]:
acc_sans_division = entraine_accumulation(diviser_loss=False)

Epoch 1, Loss : 0.369, Précision : 94.32%


Epoch 2, Loss : 0.151, Précision : 96.14%


## 9. Comparaison

In [9]:
pd.DataFrame({
    "Méthode": ["Classique, batch 32",
                "Accumulation 4 × 32 (loss divisée)",
                "Classique, batch 128",
                "Accumulation 4 × 32 (sans division)"],
    "Batch effectif": [32, 128, 128, 128],
    "Précision test (%)": [round(a * 100, 2) for a in
                           [acc_classique_32, acc_accumulation, acc_classique_128, acc_sans_division]],
}).set_index("Méthode")

,Batch effectif,Précision test (%)
Méthode,,
"Classique, batch 32",32,95.91
Accumulation 4 × 32 (loss divisée),128,95.79
"Classique, batch 128",128,95.86
Accumulation 4 × 32 (sans division),128,96.14


Ce que dit ce tableau, et ce qu'il ne dit pas :

- l'**accumulation** (loss divisée) se comporte comme le **batch de 128** qu'elle imite : on obtient une précision du même ordre, avec la mémoire d'un batch de 32 ;
- sans la division, l'entraînement est mal calibré : les mises à jour sont 4 fois trop grandes. Avec Adam, l'effet est amorti car l'optimiseur normalise les gradients, donc la différence peut sembler faible sur ce petit exemple, mais avec SGD le taux d'apprentissage effectif serait multiplié par 4 ;
- sur 2 epochs et un seul tirage, les écarts de précision entre méthodes sont petits et ne sont pas significatifs : ils changent d'une graine à l'autre. L'accumulation ne vise pas à gagner en précision, mais à **permettre** un grand batch sans la mémoire correspondante.

> ⚠️ **Attention :** l'accumulation est équivalente à un grand batch seulement pour les couches qui traitent chaque image indépendamment. Une couche `BatchNorm` calcule ses statistiques par micro-batch (ici 32), pas sur les 128 images. L'équivalence n'est donc pas parfaite dans ce cas.

## 10. Ce qu'il faut retenir

- `loss.backward()` **ajoute** les gradients à ceux déjà stockés : c'est ce qui permet d'accumuler sur plusieurs micro-batchs.
- Batch effectif = taille du micro-batch × `accumulation_steps`, pour la mémoire d'un seul micro-batch.
- Divise la loss par `accumulation_steps` avant `backward()` : sans cela, le gradient est trop grand d'autant.
- Appelle `optimizer.step()` et `zero_grad()` toutes les `accumulation_steps` étapes, et à la dernière étape de l'epoch.
- L'accumulation permet un grand batch sans la mémoire, elle n'améliore pas la précision en elle-même ; attention aux couches `BatchNorm`.

---

## Pour aller plus loin

- [Entraîne des modèles plus grands sans changer de GPU](https://www.monshotdata.com/p/gradient-checkpointing) — une autre façon d'économiser la mémoire GPU
- [Mixed precision training avec PyTorch : 2,5× plus rapide](https://www.monshotdata.com/p/mixed-precision-training-pytorch) — gagner en vitesse et en mémoire
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème